# Advanced Problems with Solutions  
## Classes That Are Both Context Managers and Iterators

This notebook develops the idea that a Python object may implement **multiple protocols at once**. In particular, a class can:

- own an external resource;
- implement `__enter__` and `__exit__`;
- implement iteration;
- expose ordinary methods and properties;
- support safe manual use as well as `with`;
- enforce a clear lifecycle;
- remain testable and maintainable.

The exercises progress from protocol fundamentals to production-style resource management.

### Best-practice themes

1. Make ownership explicit.
2. Validate lifecycle state before every resource-dependent operation.
3. Make `close()` idempotent.
4. Never rely on garbage collection for timely cleanup.
5. Prefer `csv.reader` over manually splitting CSV text.
6. Keep iterator behavior separate from iterable behavior when repeatability matters.
7. Roll back partially acquired resources.
8. Preserve the original exception unless suppression is deliberate.
9. Test early exit, exhaustion, exceptions, repeated cleanup, and invalid states.
10. On Windows, verify that files can be deleted immediately after closure.

## Notebook setup

All examples use temporary files generated by the notebook. No external dataset is required.
The final cell removes the workspace after confirming that no file handle remains open.

In [2]:
from __future__ import annotations

import asyncio
import csv
import gc
import io
import itertools
import os
import shutil
import sys
import tempfile
import threading
import time
from collections import Counter
from contextlib import (
    AbstractContextManager,
    AsyncExitStack,
    ContextDecorator,
    ExitStack,
    closing,
    contextmanager,
)
from dataclasses import dataclass
from enum import Enum, auto
from pathlib import Path
from typing import (
    Any,
    AsyncIterator,
    Callable,
    Generic,
    Iterable,
    Iterator,
    Mapping,
    Protocol,
    TextIO,
    TypeVar,
)

WORKSPACE = Path(tempfile.mkdtemp(prefix="iterator_context_manager_"))
PEOPLE_CSV = WORKSPACE / "people.csv"
MIXED_CSV = WORKSPACE / "mixed.csv"
NUMBERS_TXT = WORKSPACE / "numbers.txt"

with PEOPLE_CSV.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["id", "name", "age", "city"])
    writer.writerows([
        [1, "Ada Lovelace", 36, "London"],
        [2, "Grace Hopper", 85, "New York"],
        [3, "Edsger Dijkstra", 72, "Nuenen"],
        [4, "Barbara Liskov", 84, "Boston"],
        [5, "Donald Knuth", 88, "Stanford"],
    ])

with MIXED_CSV.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["id", "name", "age"])
    writer.writerow([1, "Alice", 31])
    writer.writerow([2, "Bob, Jr.", 27])
    writer.writerow(["bad-id", "Charlie", 40])
    writer.writerow([4, "Dana", "unknown"])
    writer.writerow([5, "Eve", 29])

with NUMBERS_TXT.open("w", encoding="utf-8") as file:
    for value in range(1, 101):
        file.write(f"{value}\n")

# print("Workspace:", WORKSPACE)
print("Fixture files:", sorted(path.name for path in WORKSPACE.iterdir()))

Fixture files: ['mixed.csv', 'numbers.txt', 'people.csv']


## Protocol map

A class can participate in several independent protocols:

| Capability | Required methods |
|---|---|
| Context manager | `__enter__`, `__exit__` |
| Iterator | `__iter__`, `__next__` |
| Iterable | `__iter__` returning an iterator |
| Closable resource | conventional `close()` method |
| Async context manager | `__aenter__`, `__aexit__` |
| Async iterator | `__aiter__`, `__anext__` |

Implementing one protocol does not prevent a class from implementing another. The important design question is not *whether* protocols can be combined, but **which object owns the resource and when that resource is valid**.

# Problem 1 — Diagnose the invalid-state failure

The basic design below stores `None` until `__enter__` opens the file. Calling `next()` before entering the context produces an obscure `TypeError`.

Design a safer version that:

- raises a meaningful `RuntimeError` before entry;
- raises a meaningful `RuntimeError` after exit;
- remains a valid iterator while its context is active;
- exposes an `is_open` property.

In [3]:
class GuardedLineIterator:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None

    @property
    def is_open(self) -> bool:
        return self._file is not None and not self._file.closed

    def __enter__(self) -> GuardedLineIterator:
        if self.is_open:
            raise RuntimeError("iterator is already open")
        self._file = self._path.open("r", encoding="utf-8")
        return self

    def __iter__(self) -> GuardedLineIterator:
        return self

    def __next__(self) -> str:
        if not self.is_open:
            raise RuntimeError(
                "iteration requires an active context: use 'with GuardedLineIterator(...)'"
            )
        assert self._file is not None
        return next(self._file).rstrip("\n")

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        return False


reader = GuardedLineIterator(NUMBERS_TXT)

try:
    next(reader)
except RuntimeError as exc:
    print("Before entry:", exc)

with reader as active:
    assert active is reader
    assert reader.is_open
    print("Inside context:", next(reader), next(reader))

assert not reader.is_open

try:
    next(reader)
except RuntimeError as exc:
    print("After exit:", exc)

Before entry: iteration requires an active context: use 'with GuardedLineIterator(...)'
Inside context: 1 2
After exit: iteration requires an active context: use 'with GuardedLineIterator(...)'


### Solution notes

A protocol error should be reported at the abstraction level of the class.  
`"'NoneType' object is not an iterator"` exposes an implementation detail; a lifecycle-specific `RuntimeError` explains the actual mistake.

`__exit__` returns `False`, so exceptions from the body are not suppressed.

# Problem 2 — Model the lifecycle as a state machine

Implement explicit states:

- `CREATED`
- `OPEN`
- `EXHAUSTED`
- `CLOSED`

Requirements:

1. Entry is allowed from `CREATED` or `CLOSED`.
2. Nested entry is rejected.
3. Reaching end-of-file changes the state to `EXHAUSTED`.
4. Exiting changes the state to `CLOSED`.
5. The object may be entered again after it has been closed.

In [4]:
class ReaderState(Enum):
    CREATED = auto()
    OPEN = auto()
    EXHAUSTED = auto()
    CLOSED = auto()


class StatefulLineReader:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None
        self._state = ReaderState.CREATED

    @property
    def state(self) -> ReaderState:
        return self._state

    def __enter__(self) -> StatefulLineReader:
        if self._state is ReaderState.OPEN:
            raise RuntimeError("nested entry is not supported")
        self._file = self._path.open("r", encoding="utf-8")
        self._state = ReaderState.OPEN
        return self

    def __iter__(self) -> StatefulLineReader:
        return self

    def __next__(self) -> str:
        if self._state is ReaderState.EXHAUSTED:
            raise StopIteration
        if self._state is not ReaderState.OPEN or self._file is None:
            raise RuntimeError(f"cannot iterate while state is {self._state.name}")
        try:
            return next(self._file).rstrip("\n")
        except StopIteration:
            self._state = ReaderState.EXHAUSTED
            raise

    def close(self) -> None:
        if self._file is not None and not self._file.closed:
            self._file.close()
        self._file = None
        self._state = ReaderState.CLOSED

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.close()
        return False


stateful = StatefulLineReader(NUMBERS_TXT)
assert stateful.state is ReaderState.CREATED

with stateful:
    assert stateful.state is ReaderState.OPEN
    first_pass = list(stateful)
    assert stateful.state is ReaderState.EXHAUSTED

assert stateful.state is ReaderState.CLOSED

with stateful:
    assert next(stateful) == "1"
assert stateful.state is ReaderState.CLOSED

print("First pass length:", len(first_pass))
print("Reusable sequential entry works.")

First pass length: 100
Reusable sequential entry works.


### Solution notes

An explicit state machine makes valid transitions reviewable and testable.  
It is preferable to inferring state from several loosely related fields.

The class is **sequentially reusable**, but not reentrant: opening the same object inside an already active context would overwrite its resource.

# Problem 3 — Support manual and contextual use

Create an iterator that supports both styles:

```python
reader = ManagedNumberReader(path)
reader.open()
try:
    ...
finally:
    reader.close()
```

and:

```python
with ManagedNumberReader(path) as reader:
    ...
```

`close()` must be idempotent, and `open()` must return `self`.

In [5]:
class ManagedNumberReader:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None

    @property
    def closed(self) -> bool:
        return self._file is None or self._file.closed

    def open(self) -> ManagedNumberReader:
        if not self.closed:
            raise RuntimeError("reader is already open")
        self._file = self._path.open("r", encoding="utf-8")
        return self

    def close(self) -> None:
        if self._file is not None and not self._file.closed:
            self._file.close()
        self._file = None

    def __enter__(self) -> ManagedNumberReader:
        return self.open()

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.close()
        return False

    def __iter__(self) -> ManagedNumberReader:
        return self

    def __next__(self) -> int:
        if self.closed:
            raise RuntimeError("reader is closed")
        assert self._file is not None
        return int(next(self._file))


manual = ManagedNumberReader(NUMBERS_TXT).open()
try:
    assert [next(manual), next(manual), next(manual)] == [1, 2, 3]
finally:
    manual.close()
    manual.close()  # idempotent

with ManagedNumberReader(NUMBERS_TXT) as contextual:
    assert sum(itertools.islice(contextual, 5)) == 15

assert contextual.closed
print("Manual and contextual use both passed.")

Manual and contextual use both passed.


### Solution notes

`__enter__` delegates to `open()` and `__exit__` delegates to `close()`.  
This keeps one authoritative implementation for acquisition and release.

Idempotent cleanup simplifies `finally` blocks and composed cleanup logic.

# Problem 4 — Roll back partial acquisition

A class needs two resources: a data file and a log file. If opening the second resource fails, the first must be closed immediately.

Implement `DualFileSession` with `ExitStack` so that:

- partial acquisition is rolled back;
- successful acquisition transfers cleanup responsibility to the object;
- both handles are closed on exit.

In [6]:
class DualFileSession:
    def __init__(self, data_path: str | Path, log_path: str | Path) -> None:
        self._data_path = Path(data_path)
        self._log_path = Path(log_path)
        self._stack: ExitStack | None = None
        self.data: TextIO | None = None
        self.log: TextIO | None = None

    def __enter__(self) -> DualFileSession:
        if self._stack is not None:
            raise RuntimeError("session is already active")

        local_stack = ExitStack()
        try:
            data = local_stack.enter_context(
                self._data_path.open("r", encoding="utf-8")
            )
            log = local_stack.enter_context(
                self._log_path.open("a", encoding="utf-8")
            )
        except BaseException:
            local_stack.close()
            raise

        self._stack = local_stack
        self.data = data
        self.log = log
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        assert self._stack is not None
        try:
            return self._stack.__exit__(exc_type, exc_value, traceback)
        finally:
            self._stack = None
            self.data = None
            self.log = None


log_path = WORKSPACE / "session.log"

with DualFileSession(PEOPLE_CSV, log_path) as session:
    assert session.data is not None
    assert session.log is not None
    first_line = session.data.readline().strip()
    session.log.write("opened successfully\n")

print("Header:", first_line)
print("Log:", log_path.read_text(encoding="utf-8").strip())

# Simulate failure after the first acquisition.
bad_log_path = WORKSPACE / "missing-directory" / "session.log"
try:
    with DualFileSession(PEOPLE_CSV, bad_log_path):
        pass
except FileNotFoundError:
    print("Partial acquisition rolled back safely.")

Header: id,name,age,city
Log: opened successfully
Partial acquisition rolled back safely.


### Solution notes

A local `ExitStack` is a transactional acquisition mechanism:

1. register each resource immediately;
2. if a later acquisition fails, close everything already registered;
3. only after all acquisitions succeed, store the stack on the object.

This pattern scales to any number of resources.

# Problem 5 — Iterator versus iterable

A self-iterator (`__iter__` returns `self`) is naturally single-pass during each active context.  
Design a **reiterable** `CSVRows` object whose `__iter__` returns a fresh iterator every time.

The container itself must not own an open file between calls.

In [7]:
class CSVRows:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)

    def __iter__(self) -> Iterator[list[str]]:
        with self._path.open("r", newline="", encoding="utf-8") as file:
            yield from csv.reader(file)


rows = CSVRows(PEOPLE_CSV)
pass_one = list(rows)
pass_two = list(rows)

assert pass_one == pass_two
assert len(pass_one) == 6
print("Two independent passes produced", len(pass_one), "rows each.")

Two independent passes produced 6 rows each.


### Solution notes

`CSVRows` is an **iterable**, not an iterator. Each call to `iter(rows)` creates a generator with its own file handle. The generator's `with` block keeps the resource open for exactly the duration of iteration.

This design is often preferable when users reasonably expect repeated iteration.

# Problem 6 — Reject nested use but allow sequential reuse

Build `ReusableCSVSession` that:

- can be entered multiple times sequentially;
- rejects nested use of the same object;
- resets its row counter on every entry;
- reports the number of rows consumed in the most recent session.

In [8]:
class ReusableCSVSession:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None
        self._reader: Iterator[list[str]] | None = None
        self._active = False
        self.rows_read = 0

    def __enter__(self) -> ReusableCSVSession:
        if self._active:
            raise RuntimeError("the same session cannot be nested")
        self._file = self._path.open("r", newline="", encoding="utf-8")
        self._reader = csv.reader(self._file)
        self._active = True
        self.rows_read = 0
        return self

    def __iter__(self) -> ReusableCSVSession:
        return self

    def __next__(self) -> list[str]:
        if not self._active or self._reader is None:
            raise RuntimeError("session is not active")
        row = next(self._reader)
        self.rows_read += 1
        return row

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        try:
            if self._file is not None:
                self._file.close()
        finally:
            self._file = None
            self._reader = None
            self._active = False
        return False


session = ReusableCSVSession(PEOPLE_CSV)

with session:
    assert len(list(session)) == 6
assert session.rows_read == 6

with session:
    assert len(list(itertools.islice(session, 2))) == 2
assert session.rows_read == 2

with session:
    try:
        with session:
            pass
    except RuntimeError as exc:
        print("Nested entry rejected:", exc)

print("Sequential reuse passed.")

Nested entry rejected: the same session cannot be nested
Sequential reuse passed.


### Solution notes

Reusability and reentrancy are different properties:

- **Reusable:** the object can be used again after the previous session finishes.
- **Reentrant:** the object can be entered again while already active.

A single object holding one mutable file slot is usually reusable but not reentrant.

# Problem 7 — Parse CSV correctly

The source example used `line.strip().split(",")`, which fails for quoted commas.

Implement a context-managed iterator based on `csv.reader` and demonstrate that the name `"Bob, Jr."` remains one field.

In [9]:
class CSVIterator:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None
        self._reader: Iterator[list[str]] | None = None

    def __enter__(self) -> CSVIterator:
        self._file = self._path.open("r", newline="", encoding="utf-8")
        self._reader = csv.reader(self._file)
        return self

    def __iter__(self) -> CSVIterator:
        return self

    def __next__(self) -> list[str]:
        if self._reader is None:
            raise RuntimeError("CSVIterator must be used inside a with statement")
        return next(self._reader)

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        self._file = None
        self._reader = None
        return False


with CSVIterator(MIXED_CSV) as rows:
    header = next(rows)
    alice = next(rows)
    bob = next(rows)

assert bob == ["2", "Bob, Jr.", "27"]
print("Header:", header)
print("Quoted field parsed correctly:", bob)

Header: ['id', 'name', 'age']
Quoted field parsed correctly: ['2', 'Bob, Jr.', '27']


### Solution notes

CSV is a format, not merely text separated by commas. Quoting, escaped quotes, embedded delimiters, and newline handling require a real parser.

For file-based CSV processing, use `newline=""` as recommended by Python's `csv` module.

# Problem 8 — Return typed domain objects

Create a context-managed iterator that:

1. reads the header;
2. converts each row to a `Person` dataclass;
3. validates required columns;
4. raises a clear error if the schema is missing a field.

In [10]:
@dataclass(frozen=True, slots=True)
class Person:
    id: int
    name: str
    age: int
    city: str


class PersonReader:
    REQUIRED = {"id", "name", "age", "city"}

    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None
        self._reader: csv.DictReader[str] | None = None

    def __enter__(self) -> PersonReader:
        self._file = self._path.open("r", newline="", encoding="utf-8")
        self._reader = csv.DictReader(self._file)
        actual = set(self._reader.fieldnames or [])
        missing = self.REQUIRED - actual
        if missing:
            self._file.close()
            self._file = None
            self._reader = None
            raise ValueError(f"missing required columns: {sorted(missing)}")
        return self

    def __iter__(self) -> PersonReader:
        return self

    def __next__(self) -> Person:
        if self._reader is None:
            raise RuntimeError("reader is not active")
        row = next(self._reader)
        return Person(
            id=int(row["id"]),
            name=row["name"],
            age=int(row["age"]),
            city=row["city"],
        )

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        self._file = None
        self._reader = None
        return False


with PersonReader(PEOPLE_CSV) as people:
    typed_people = list(people)

assert typed_people[0] == Person(1, "Ada Lovelace", 36, "London")
print("Oldest:", max(typed_people, key=lambda person: person.age))

Oldest: Person(id=5, name='Donald Knuth', age=88, city='Stanford')


### Solution notes

Resource acquisition includes setup steps such as reading and validating a header. If setup fails, `__enter__` must close the resource before re-raising.

Returning typed objects moves conversion and validation to the boundary of the system.

# Problem 9 — Add strict and tolerant error policies

Implement a reader for `MIXED_CSV` with two modes:

- `strict=True`: raise `RowConversionError` on the first malformed row;
- `strict=False`: skip malformed rows and collect diagnostics.

Diagnostics must include the physical CSV line number and the original row.

In [11]:
@dataclass(frozen=True, slots=True)
class SimplePerson:
    id: int
    name: str
    age: int


class RowConversionError(ValueError):
    def __init__(self, line_number: int, row: Mapping[str, str], reason: str) -> None:
        self.line_number = line_number
        self.row = dict(row)
        super().__init__(f"line {line_number}: {reason}; row={dict(row)!r}")


class PolicyPersonReader:
    def __init__(self, path: str | Path, *, strict: bool = True) -> None:
        self._path = Path(path)
        self._strict = strict
        self._file: TextIO | None = None
        self._reader: csv.DictReader[str] | None = None
        self.errors: list[RowConversionError] = []

    def __enter__(self) -> PolicyPersonReader:
        self._file = self._path.open("r", newline="", encoding="utf-8")
        self._reader = csv.DictReader(self._file)
        self.errors.clear()
        return self

    def __iter__(self) -> PolicyPersonReader:
        return self

    def __next__(self) -> SimplePerson:
        if self._reader is None:
            raise RuntimeError("reader is not active")

        while True:
            row = next(self._reader)
            try:
                return SimplePerson(
                    id=int(row["id"]),
                    name=row["name"],
                    age=int(row["age"]),
                )
            except (TypeError, ValueError, KeyError) as exc:
                error = RowConversionError(
                    self._reader.line_num,
                    row,
                    str(exc),
                )
                if self._strict:
                    raise error from exc
                self.errors.append(error)

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        self._file = None
        self._reader = None
        return False


try:
    with PolicyPersonReader(MIXED_CSV, strict=True) as strict_reader:
        list(strict_reader)
except RowConversionError as exc:
    print("Strict mode stopped:", exc)

with PolicyPersonReader(MIXED_CSV, strict=False) as tolerant_reader:
    valid_people = list(tolerant_reader)
    diagnostics = list(tolerant_reader.errors)

assert [person.id for person in valid_people] == [1, 2, 5]
assert len(diagnostics) == 2
print("Tolerant result:", valid_people)
print("Diagnostics:", *diagnostics, sep="\n- ")

Strict mode stopped: line 4: invalid literal for int() with base 10: 'bad-id'; row={'id': 'bad-id', 'name': 'Charlie', 'age': '40'}
Tolerant result: [SimplePerson(id=1, name='Alice', age=31), SimplePerson(id=2, name='Bob, Jr.', age=27), SimplePerson(id=5, name='Eve', age=29)]
Diagnostics:
- line 4: invalid literal for int() with base 10: 'bad-id'; row={'id': 'bad-id', 'name': 'Charlie', 'age': '40'}
- line 5: invalid literal for int() with base 10: 'unknown'; row={'id': '4', 'name': 'Dana', 'age': 'unknown'}


### Solution notes

An error policy belongs at the boundary where raw data becomes typed data.

A tolerant iterator must continue its internal loop until it either:

- produces one valid value, or
- reaches end-of-input.

It should not return `None` for a malformed record unless `None` is explicitly part of the public data model.

# Problem 10 — Preserve exceptions and suppress only deliberately

Create a context manager that counts exceptions by type.

Behavior:

- always record exceptions raised in the body;
- suppress only exceptions listed in `suppress`;
- never suppress `KeyboardInterrupt`, `SystemExit`, or other `BaseException` subclasses accidentally.

In [12]:
class ExceptionCounter(AbstractContextManager["ExceptionCounter"]):
    def __init__(self, *suppress: type[Exception]) -> None:
        self._suppress = suppress
        self.counts: Counter[str] = Counter()

    def __enter__(self) -> ExceptionCounter:
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            return False

        self.counts[exc_type.__name__] += 1

        # exc_type is guaranteed to be an exception class here.
        return issubclass(exc_type, self._suppress)


counter = ExceptionCounter(ValueError)

with counter:
    raise ValueError("suppressed intentionally")

try:
    with counter:
        raise RuntimeError("must propagate")
except RuntimeError:
    pass

assert counter.counts == Counter({"ValueError": 1, "RuntimeError": 1})
print(counter.counts)

Counter({'ValueError': 1, 'RuntimeError': 1})


### Solution notes

Returning a truthy value from `__exit__` suppresses the active exception.  
The default safe behavior is `False`.

Suppression should be narrow, documented, and testable. Catching or suppressing `BaseException` is almost never appropriate for ordinary application control flow.

# Problem 11 — Reproduce `with` manually

Demonstrate the conceptual behavior of:

```python
with manager as value:
    body(value)
```

by explicitly calling `__enter__` and `__exit__`.

The solution must pass real exception information to `__exit__` and re-raise when suppression is false.

In [13]:
class TraceManager:
    def __init__(self, events: list[str], suppress: bool = False) -> None:
        self.events = events
        self.suppress = suppress

    def __enter__(self) -> str:
        self.events.append("enter")
        return "resource"

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        label = "none" if exc_type is None else exc_type.__name__
        self.events.append(f"exit:{label}")
        return self.suppress


def run_manually(manager: TraceManager, body: Callable[[str], None]) -> None:
    value = manager.__enter__()
    try:
        body(value)
    except BaseException:
        should_suppress = manager.__exit__(*sys.exc_info())
        if not should_suppress:
            raise
    else:
        manager.__exit__(None, None, None)


events: list[str] = []
run_manually(TraceManager(events), lambda value: events.append(value))
assert events == ["enter", "resource", "exit:none"]

events = []
try:
    run_manually(
        TraceManager(events),
        lambda value: (_ for _ in ()).throw(ValueError(value)),
    )
except ValueError:
    pass

assert events == ["enter", "exit:ValueError"]
print("Manual protocol events:", events)

Manual protocol events: ['enter', 'exit:ValueError']


### Solution notes

The actual language implementation has additional details, but this captures the essential control flow:

- call `__enter__`;
- run the body;
- on failure, call `__exit__` with exception information;
- re-raise unless `__exit__` suppresses;
- on success, call `__exit__(None, None, None)`.

# Problem 12 — Use `@contextmanager` for a generator-based resource

Implement `open_number_stream(path)` with `@contextmanager`.

It should yield a lazy iterator of integers while keeping the underlying file open, then close the file when the context exits.

In [14]:
@contextmanager
def open_number_stream(path: str | Path) -> Iterator[Iterator[int]]:
    with Path(path).open("r", encoding="utf-8") as file:
        yield (int(line) for line in file)


with open_number_stream(NUMBERS_TXT) as numbers:
    first_ten_even = list(
        itertools.islice((number for number in numbers if number % 2 == 0), 10)
    )

assert first_ten_even == list(range(2, 21, 2))
print(first_ten_even)

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


### Solution notes

The `yield` divides acquisition from cleanup:

- code before `yield` behaves like `__enter__`;
- the yielded object is bound after `as`;
- code after `yield` behaves like `__exit__`.

The lazy iterator must be consumed inside the `with` block because it depends on the open file.

# Problem 13 — Manage a dynamic number of files

Use `ExitStack` to open an arbitrary list of files and iterate over all of their lines as one lazy stream.

The helper must close every file if:

- all data is consumed;
- the caller stops early;
- opening a later file fails;
- processing raises an exception.

In [15]:
@contextmanager
def open_concatenated(paths: Iterable[str | Path]) -> Iterator[Iterator[str]]:
    with ExitStack() as stack:
        files = [
            stack.enter_context(Path(path).open("r", encoding="utf-8"))
            for path in paths
        ]
        yield (
            line.rstrip("\n")
            for file in files
            for line in file
        )


extra_numbers = WORKSPACE / "extra_numbers.txt"
extra_numbers.write_text("101\n102\n103\n", encoding="utf-8")

with open_concatenated([NUMBERS_TXT, extra_numbers]) as lines:
    tail = list(itertools.islice(lines, 98, 103))

assert tail == ["99", "100", "101", "102", "103"]
print("Boundary across files:", tail)

Boundary across files: ['99', '100', '101', '102', '103']


### Solution notes

`ExitStack` is the standard tool when the number of context managers is data-dependent. It provides all-or-nothing acquisition and reverse-order cleanup.

# Problem 14 — Build an object usable as both context manager and decorator

Use `ContextDecorator` to implement `Timer`.

The same class must support:

```python
with Timer("block"):
    ...
```

and:

```python
@Timer("function")
def work():
    ...
```

Store elapsed durations in a class-level registry.

In [16]:
class Timer(ContextDecorator):
    registry: dict[str, list[float]] = {}

    def __init__(self, label: str) -> None:
        self.label = label
        self._start: float | None = None

    def __enter__(self) -> Timer:
        if self._start is not None:
            raise RuntimeError("timer is already running")
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        assert self._start is not None
        elapsed = time.perf_counter() - self._start
        self.registry.setdefault(self.label, []).append(elapsed)
        self._start = None
        return False


with Timer("block"):
    sum(range(10_000))


@Timer("function")
def compute() -> int:
    return sum(value * value for value in range(1_000))


assert compute() > 0
assert len(Timer.registry["block"]) == 1
assert len(Timer.registry["function"]) == 1
print({label: len(values) for label, values in Timer.registry.items()})

{'block': 1, 'function': 1}


### Solution notes

This is another example of one class implementing more than one useful role. `ContextDecorator` converts context-manager behavior into decorator behavior while preserving exception propagation.

# Problem 15 — Adapt a closable non-context-manager

A third-party object exposes `close()` but does not implement `__enter__` or `__exit__`.

Use `contextlib.closing` to guarantee cleanup.

In [17]:
class ThirdPartyCursor:
    def __init__(self, values: Iterable[int]) -> None:
        self._iterator = iter(values)
        self.closed = False

    def __iter__(self) -> ThirdPartyCursor:
        return self

    def __next__(self) -> int:
        if self.closed:
            raise RuntimeError("cursor is closed")
        return next(self._iterator)

    def close(self) -> None:
        self.closed = True


cursor = ThirdPartyCursor(range(10))

with closing(cursor) as active_cursor:
    prefix = list(itertools.islice(active_cursor, 4))

assert prefix == [0, 1, 2, 3]
assert cursor.closed
print("Adapted cursor closed:", cursor.closed)

Adapted cursor closed: True


### Solution notes

`closing(obj)` is appropriate when `obj.close()` is the correct cleanup action but the object does not implement the context-manager protocol itself.

# Problem 16 — Use dependency injection for testable I/O

Implement a reader that accepts an `opener` dependency.  
Test it with an in-memory `StringIO` rather than a real file.

The reader must still close the injected stream on exit.

In [18]:
Opener = Callable[[Path], TextIO]


class InjectableLineReader:
    def __init__(self, path: str | Path, opener: Opener) -> None:
        self._path = Path(path)
        self._opener = opener
        self._file: TextIO | None = None

    def __enter__(self) -> InjectableLineReader:
        self._file = self._opener(self._path)
        return self

    def __iter__(self) -> InjectableLineReader:
        return self

    def __next__(self) -> str:
        if self._file is None:
            raise RuntimeError("reader is not active")
        return next(self._file).rstrip("\n")

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        return False


fake_stream = io.StringIO("alpha\nbeta\ngamma\n")

def fake_opener(path: Path) -> TextIO:
    assert path.name == "virtual.txt"
    return fake_stream


with InjectableLineReader("virtual.txt", fake_opener) as reader:
    assert list(reader) == ["alpha", "beta", "gamma"]

assert fake_stream.closed
print("In-memory dependency was closed.")

In-memory dependency was closed.


### Solution notes

Dependency injection separates resource policy from resource creation.  
It enables fast tests and supports alternate sources such as compressed streams, network wrappers, or encrypted files.

# Problem 17 — Guard against concurrent `next()` calls

A file iterator is not automatically safe for concurrent consumption.

Create `LockedLineIterator` that serializes `next()` calls with a lock and returns each line exactly once when several threads consume the same active iterator.

In [19]:
class LockedLineIterator:
    def __init__(self, path: str | Path) -> None:
        self._path = Path(path)
        self._file: TextIO | None = None
        self._lock = threading.Lock()

    def __enter__(self) -> LockedLineIterator:
        self._file = self._path.open("r", encoding="utf-8")
        return self

    def __iter__(self) -> LockedLineIterator:
        return self

    def __next__(self) -> int:
        with self._lock:
            if self._file is None:
                raise RuntimeError("iterator is not active")
            return int(next(self._file))

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self._file is not None:
            self._file.close()
        self._file = None
        return False


def consume_shared(iterator: Iterator[int], output: list[int], output_lock: threading.Lock) -> None:
    while True:
        try:
            value = next(iterator)
        except StopIteration:
            return
        with output_lock:
            output.append(value)


collected: list[int] = []
output_lock = threading.Lock()

with LockedLineIterator(NUMBERS_TXT) as shared:
    threads = [
        threading.Thread(target=consume_shared, args=(shared, collected, output_lock))
        for _ in range(4)
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

assert sorted(collected) == list(range(1, 101))
assert len(collected) == len(set(collected)) == 100
print("Concurrent consumers received 100 unique values.")

Concurrent consumers received 100 unique values.


### Solution notes

The lock protects the critical operation that advances shared mutable state.  
This makes `next()` atomic with respect to other threads, but it does not make arbitrary methods or lifecycle transitions thread-safe.

Often the better architecture is one owner thread plus a queue. This exercise focuses on protocol-level synchronization.

# Problem 18 — Combine async context management and async iteration

Implement an asynchronous line source that:

- opens logically in `__aenter__`;
- yields values through `__anext__`;
- simulates asynchronous I/O with `await asyncio.sleep(0)`;
- closes in `__aexit__`;
- rejects iteration outside an active async context.

In [20]:
class AsyncSequence:
    def __init__(self, values: Iterable[int]) -> None:
        self._values = list(values)
        self._iterator: Iterator[int] | None = None
        self.closed = True

    async def __aenter__(self) -> AsyncSequence:
        if not self.closed:
            raise RuntimeError("sequence is already active")
        await asyncio.sleep(0)
        self._iterator = iter(self._values)
        self.closed = False
        return self

    def __aiter__(self) -> AsyncSequence:
        return self

    async def __anext__(self) -> int:
        if self.closed or self._iterator is None:
            raise RuntimeError("async iteration requires an active async context")
        await asyncio.sleep(0)
        try:
            return next(self._iterator)
        except StopIteration:
            raise StopAsyncIteration

    async def __aexit__(self, exc_type, exc_value, traceback) -> bool:
        await asyncio.sleep(0)
        self._iterator = None
        self.closed = True
        return False


async def async_sequence_demo() -> list[int]:
    source = AsyncSequence(range(1, 8))
    result: list[int] = []
    async with source:
        async for value in source:
            if value % 2:
                result.append(value)
    assert source.closed
    return result


async_result = await async_sequence_demo()
assert async_result == [1, 3, 5, 7]
print("Async result:", async_result)

Async result: [1, 3, 5, 7]


### Solution notes

Async protocols mirror their synchronous counterparts but allow acquisition, iteration, and cleanup to await nonblocking operations.

`StopIteration` must be translated to `StopAsyncIteration` inside `__anext__`.

# Problem 19 — Compose async resources dynamically

Use `AsyncExitStack` to enter several asynchronous sequences and guarantee reverse-order cleanup if processing fails.

In [21]:
async def async_stack_demo() -> tuple[list[int], list[bool]]:
    sources = [AsyncSequence([1, 2]), AsyncSequence([10, 20]), AsyncSequence([100])]
    combined: list[int] = []

    async with AsyncExitStack() as stack:
        active_sources = [
            await stack.enter_async_context(source)
            for source in sources
        ]
        for source in active_sources:
            async for value in source:
                combined.append(value)

    return combined, [source.closed for source in sources]


combined_async, closed_flags = await async_stack_demo()
assert combined_async == [1, 2, 10, 20, 100]
assert all(closed_flags)
print("Combined:", combined_async)
print("Closed:", closed_flags)

Combined: [1, 2, 10, 20, 100]
Closed: [True, True, True]


### Solution notes

`AsyncExitStack` is the asynchronous analogue of `ExitStack`. It is useful when the number or type of async resources is not known until runtime.

# Problem 20 — Capstone: production-style managed CSV stream

Implement `ManagedCSV` with the following contract:

### Lifecycle

- sequentially reusable;
- not reentrant;
- explicit `open()` and idempotent `close()`;
- usable with or without `with`;
- safe after partial consumption;
- no leaked file handles.

### Iteration

- uses `csv.DictReader`;
- optionally transforms each row;
- optionally filters transformed values;
- tracks physical rows read, values yielded, and rows rejected;
- supports strict and tolerant conversion policies.

### API

- `active`
- `fieldnames`
- `stats`
- `errors`
- `open()`
- `close()`
- iterator protocol
- context-manager protocol

Then test normal completion, early exit, conversion errors, reuse, manual use, and immediate file deletion on Windows-style semantics.

In [22]:
T = TypeVar("T")


@dataclass(frozen=True, slots=True)
class StreamStats:
    physical_rows: int
    yielded: int
    rejected: int


class ManagedCSV(Generic[T]):
    def __init__(
        self,
        path: str | Path,
        *,
        transform: Callable[[dict[str, str]], T],
        predicate: Callable[[T], bool] | None = None,
        strict: bool = True,
    ) -> None:
        self._path = Path(path)
        self._transform = transform
        self._predicate = predicate or (lambda value: True)
        self._strict = strict

        self._file: TextIO | None = None
        self._reader: csv.DictReader[str] | None = None
        self._active = False
        self._fieldnames: tuple[str, ...] = ()
        self._physical_rows = 0
        self._yielded = 0
        self._rejected = 0
        self._errors: list[RowConversionError] = []

    @property
    def active(self) -> bool:
        return self._active

    @property
    def fieldnames(self) -> tuple[str, ...]:
        return self._fieldnames

    @property
    def stats(self) -> StreamStats:
        return StreamStats(
            physical_rows=self._physical_rows,
            yielded=self._yielded,
            rejected=self._rejected,
        )

    @property
    def errors(self) -> tuple[RowConversionError, ...]:
        return tuple(self._errors)

    def open(self) -> ManagedCSV[T]:
        if self._active:
            raise RuntimeError("ManagedCSV is already active")

        file = self._path.open("r", newline="", encoding="utf-8")
        try:
            reader = csv.DictReader(file)
            fieldnames = tuple(reader.fieldnames or ())
            if not fieldnames:
                raise ValueError("CSV file has no header")
        except BaseException:
            file.close()
            raise

        self._file = file
        self._reader = reader
        self._fieldnames = fieldnames
        self._physical_rows = 0
        self._yielded = 0
        self._rejected = 0
        self._errors.clear()
        self._active = True
        return self

    def close(self) -> None:
        try:
            if self._file is not None and not self._file.closed:
                self._file.close()
        finally:
            self._file = None
            self._reader = None
            self._active = False

    def __enter__(self) -> ManagedCSV[T]:
        return self.open()

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.close()
        return False

    def __iter__(self) -> ManagedCSV[T]:
        return self

    def __next__(self) -> T:
        if not self._active or self._reader is None:
            raise RuntimeError("ManagedCSV is not active")

        while True:
            row = next(self._reader)
            self._physical_rows += 1

            try:
                value = self._transform(dict(row))
            except (TypeError, ValueError, KeyError) as exc:
                self._rejected += 1
                error = RowConversionError(
                    self._reader.line_num,
                    row,
                    str(exc),
                )
                if self._strict:
                    raise error from exc
                self._errors.append(error)
                continue

            if not self._predicate(value):
                self._rejected += 1
                continue

            self._yielded += 1
            return value


def to_simple_person(row: dict[str, str]) -> SimplePerson:
    return SimplePerson(
        id=int(row["id"]),
        name=row["name"],
        age=int(row["age"]),
    )


# 1. Tolerant conversion and filtering.
stream = ManagedCSV(
    MIXED_CSV,
    transform=to_simple_person,
    predicate=lambda person: person.age >= 29,
    strict=False,
)

with stream:
    selected = list(stream)
    first_stats = stream.stats
    first_errors = stream.errors

assert [person.name for person in selected] == ["Alice", "Eve"]
assert first_stats == StreamStats(5, 2, 3)
assert len(first_errors) == 2
assert not stream.active

# 2. Sequential reuse resets counters.
with stream:
    prefix = list(itertools.islice(stream, 1))
    second_stats = stream.stats

assert prefix == [SimplePerson(1, "Alice", 31)]
assert second_stats == StreamStats(1, 1, 0)

# 3. Manual use and idempotent close.
manual_stream = ManagedCSV(
    PEOPLE_CSV,
    transform=lambda row: row["name"],
)
manual_stream.open()
try:
    assert next(manual_stream) == "Ada Lovelace"
finally:
    manual_stream.close()
    manual_stream.close()

# 4. Early body exception still closes the resource.
try:
    with ManagedCSV(PEOPLE_CSV, transform=lambda row: row) as failing:
        next(failing)
        raise LookupError("simulated processing failure")
except LookupError:
    pass

assert not failing.active

# 5. Verify immediate deletion after closure.
deletion_probe = WORKSPACE / "deletion_probe.csv"
deletion_probe.write_text("x\n1\n2\n", encoding="utf-8")

with ManagedCSV(deletion_probe, transform=lambda row: int(row["x"])) as probe:
    assert next(probe) == 1

deletion_probe.unlink()
assert not deletion_probe.exists()

print("Selected:", selected)
print("First session stats:", first_stats)
print("Second session stats:", second_stats)
print("Immediate post-close deletion passed.")

Selected: [SimplePerson(id=1, name='Alice', age=31), SimplePerson(id=5, name='Eve', age=29)]
First session stats: StreamStats(physical_rows=5, yielded=2, rejected=3)
Second session stats: StreamStats(physical_rows=1, yielded=1, rejected=0)
Immediate post-close deletion passed.


### Capstone discussion

The capstone separates four concerns:

1. **Ownership:** `ManagedCSV` owns the file only while active.
2. **Parsing:** `DictReader` handles CSV syntax and headers.
3. **Conversion:** a caller-supplied transform creates domain values.
4. **Selection:** a caller-supplied predicate filters converted values.

Important invariants:

- `_active` implies both `_file` and `_reader` exist.
- `close()` clears all resource references even if cleanup itself raises.
- setup failures close the local file before object state is committed.
- metrics reset on every new session.
- early exit is safe because context exit closes the file.
- immediate deletion tests timely cleanup rather than eventual garbage collection.

# Additional challenge drills

These are intentionally left as extension tasks after the solved set.

1. Add checkpoint/resume support using `file.tell()` and `file.seek()`.
2. Add a `max_errors` threshold to tolerant mode.
3. Support gzip files through an injected opener.
4. Implement a fresh-iterator container whose child iterators are context managers.
5. Add schema aliases such as `{"years": "age"}`.
6. Record processing latency per row.
7. Add an async transform to the async iterator example.
8. Use `weakref.finalize` only as a diagnostic fallback, not primary cleanup.
9. Add property-based tests for lifecycle transitions.
10. Define a `Protocol` for closable iterators and write a generic adapter.

# Final verification and cleanup

The following checks cover the most important resource-safety properties:

- all reusable readers are inactive;
- no temporary fixture file is locked;
- cleanup is explicit and deterministic;
- the notebook does not depend on garbage collection for ordinary closure.

In [23]:
assert not stream.active
assert not manual_stream.active
assert not failing.active
assert contextual.closed
assert stateful.state is ReaderState.CLOSED
assert not session._active

# Force finalization only as an extra diagnostic step.
gc.collect()

# On Windows, this removal would fail if any example still held an open handle.
shutil.rmtree(WORKSPACE)
assert not WORKSPACE.exists()

print("All solved problems passed.")
print("Temporary workspace removed successfully.")

All solved problems passed.
Temporary workspace removed successfully.
